In [0]:
dbutils.library.restartPython()

In [0]:
%pip install catboost

In [0]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

import io
import base64

def mostrar_grafica():
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()
    plt.close()
    displayHTML(f'<img src="data:image/png;base64,{img_base64}"/>')

print("CatBoost importado correctamente")

In [0]:
TABLA = "workspace.tesis.enoe_jovenes_cdmx"
df = spark.table(TABLA).toPandas()

print(f"Filas    : {df.shape[0]:,}")
print(f"Columnas : {df.shape[1]}")

## Preparación de los datos

In [0]:
FEATURES = [
    'eda',
    'sex',
    'anios_esc',
    'n_hij',
    'niv_ins',
    'e_con',
    'periodo_ord',
    'grupo_edad'
]

TARGET = 'desempleado'

df_model = df[df['pea'] == 1][FEATURES + [TARGET]].copy()
df_model['n_hij'] = df_model['n_hij'].fillna(0)
df_model = df_model.dropna()

print(f"Filas para modelado : {df_model.shape[0]:,}")
print(f"Desempleados        : {df_model[TARGET].sum():,} ({df_model[TARGET].mean()*100:.1f}%)")
print(f"Empleados           : {(df_model[TARGET]==0).sum():,} ({(df_model[TARGET]==0).mean()*100:.1f}%)")

In [0]:
X = df_model[FEATURES]
y = df_model[TARGET]

# Variables categóricas para CatBoost
CAT_FEATURES = ['grupo_edad', 'e_con']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]:,} filas")
print(f"Test  : {X_test.shape[0]:,} filas")

In [0]:
modelo = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    scale_pos_weight=y_train.value_counts()[0] / y_train.value_counts()[1],
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    cat_features=CAT_FEATURES
)

modelo.fit(
    X_train, y_train,
    eval_set=(X_test, y_test)
)

In [0]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred, target_names=['Empleado', 'Desempleado']))

print(f"\nAUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

In [0]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', label='Azar')
plt.title('Curva ROC — Modelo de desempleo juvenil')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.legend()
plt.tight_layout()
mostrar_grafica()

In [0]:
importancias = pd.DataFrame({
    'variable': FEATURES,
    'importancia': modelo.get_feature_importance()
}).sort_values('importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importancias, x='importancia', y='variable', color='steelblue')
plt.title('Importancia de variables — Modelo CatBoost')
plt.xlabel('Importancia')
plt.ylabel('')
plt.tight_layout()
mostrar_grafica()

##Conclusiones

### Desempeño
El modelo CatBoost entrenado con variables sociodemográficas obtiene un
AUC-ROC de 0.6526, superior al azar (0.50) pero con limitaciones inherentes
a la naturaleza del problema. Predecir desempleo individual con características
personales es difícil porque el desempleo tiene un componente estructural y
coyuntural que las variables sociodemográficas no capturan completamente.

### Variables más importantes
1. `e_con` (estado conyugal) — la variable con mayor poder predictivo,
   sugiere que el estado civil está fuertemente asociado a la participación
   y estabilidad laboral en jóvenes
2. `periodo_ord` — el período temporal es la segunda variable más importante,
   confirma que el contexto macroeconómico (pandemia, recuperación) determina
   el desempleo más que las características individuales
3. `anios_esc` y `eda` — escolaridad y edad tienen peso similar,
   consistente con los hallazgos del EDA

### Limitaciones
- AUC de 0.65 indica capacidad predictiva moderada
- Con variables macroeconómicas adicionales (PIB, inflación, salario mínimo)
  el modelo podría mejorar significativamente
- El desbalance 90/10 limita la precisión en la clase minoritaria

### Siguiente paso
Integrar variables macroeconómicas del Banco de México e INEGI para
mejorar el poder predictivo del modelo.
